**Imports and Setup**

*This cell sets up the environment and defines the Permuted MNIST transformation.*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
import copy
from tqdm.auto import tqdm  # Add this to your imports

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Function to create Permuted MNIST
def get_permute_mnist(permutation):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
        transforms.Lambda(lambda x: x.view(-1)[permutation].view(1, 28, 28))
    ])
    return transform

# 1. Prepare Data
batch_size = 64
# Task 1: Standard MNIST
t1_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset_t1 = datasets.MNIST(root='./data', train=True, download=True, transform=t1_transform)
test_dataset_t1 = datasets.MNIST(root='./data', train=False, download=True, transform=t1_transform)

# Task 2: Permuted MNIST
pixel_permutation = torch.randperm(784) # Fix the shuffle
t2_transform = get_permute_mnist(pixel_permutation)
train_dataset_t2 = datasets.MNIST(root='./data', train=True, download=True, transform=t2_transform)
test_dataset_t2 = datasets.MNIST(root='./data', train=False, download=True, transform=t2_transform)

train_loader_t1 = DataLoader(train_dataset_t1, batch_size=batch_size, shuffle=True)
test_loader_t1 = DataLoader(test_dataset_t1, batch_size=batch_size, shuffle=False)
train_loader_t2 = DataLoader(train_dataset_t2, batch_size=batch_size, shuffle=True)
test_loader_t2 = DataLoader(test_dataset_t2, batch_size=batch_size, shuffle=False)

Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 344kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.21MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.89MB/s]


**The CNN Model & Evaluation Metrics**

*We define a simple CNN and the logic for Average Accuracy and Forgetting.*

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            correct += (output.argmax(1) == target).sum().item()
    return 100 * correct / len(loader.dataset)

**Strategy 1 - Naive Approach (The Baseline)**

This shows what happens when you do nothing to stop forgetting.

In [ ]:
print("--- Starting Naive Training ---")
model_naive = SimpleCNN().to(device)
optimizer = optim.Adam(model_naive.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training Task 1
print("\n[Task 1: Standard MNIST]")
for epoch in range(1, 3):
    model_naive.train()
    pbar = tqdm(train_loader_t1, unit="batch")
    pbar.set_description(f"Epoch {epoch}/2")

    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model_naive(data), target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"✓ Epoch {epoch} finished.")

# Training Task 2
print("\n[Task 2: Permuted MNIST]")
for epoch in range(1, 3):
    model_naive.train()
    pbar = tqdm(train_loader_t2, unit="batch")
    pbar.set_description(f"Epoch {epoch}/2")

    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model_naive(data), target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"✓ Epoch {epoch} finished.")

acc1_after_t2_naive = evaluate(model_naive, test_loader_t1)
print(f"\nNaive Strategy Complete. Task 1 Final Accuracy: {acc1_after_t2_naive:.2f}%")

--- Starting Naive Training ---

[Task 1: Standard MNIST]


  0%|          | 0/938 [00:00<?, ?batch/s]

✓ Epoch 1 finished.


  0%|          | 0/938 [00:00<?, ?batch/s]

✓ Epoch 2 finished.

[Task 2: Permuted MNIST]


  0%|          | 0/938 [00:00<?, ?batch/s]

✓ Epoch 1 finished.


  0%|          | 0/938 [00:00<?, ?batch/s]

✓ Epoch 2 finished.

Naive Strategy Complete. Task 1 Final Accuracy: 71.92%


**Strategy 2 - Experience Replay (Rehearsal)**

We store a few images from Task 1 and mix them into Task 2 training.

In [ ]:
print("\n--- Starting Experience Replay ---")
model_replay = SimpleCNN().to(device)
optimizer = optim.Adam(model_replay.parameters(), lr=0.001)

# Train Task 1
print("\n[Task 1: Standard MNIST]")
for epoch in range(1, 3):
    pbar = tqdm(train_loader_t1, unit="batch")
    pbar.set_description(f"Epoch {epoch}/2")
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model_replay(data), target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    print(f"✓ Epoch {epoch} finished.")

# Train Task 2 with Replay
print("\n[Task 2: Permuted MNIST + Replay]")
buffer_iter = iter(buffer_loader)
for epoch in range(1, 3):
    pbar = tqdm(train_loader_t2, unit="batch")
    pbar.set_description(f"Epoch {epoch}/2")
    for data, target in pbar:
        try:
            b_data, b_target = next(buffer_iter)
        except StopIteration:
            buffer_iter = iter(buffer_loader)
            b_data, b_target = next(buffer_iter)

        data, target = data.to(device), target.to(device)
        b_data, b_target = b_data.to(device), b_target.to(device)

        optimizer.zero_grad()
        # Combine losses (New data + Memory buffer)
        loss = criterion(model_replay(data), target) + criterion(model_replay(b_data), b_target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    print(f"✓ Epoch {epoch} finished.")

acc1_after_t2_replay = evaluate(model_replay, test_loader_t1)
print(f"\nReplay Strategy Complete. Task 1 Final Accuracy: {acc1_after_t2_replay:.2f}%")


--- Starting Experience Replay ---

[Task 1: Standard MNIST]


  0%|          | 0/938 [00:00<?, ?batch/s]

✓ Epoch 1 finished.


  0%|          | 0/938 [00:00<?, ?batch/s]

✓ Epoch 2 finished.

[Task 2: Permuted MNIST + Replay]


NameError: name 'buffer_loader' is not defined

**Strategy 3 - EWC (Regularization)**

This penalizes the change of important weights.

In [ ]:
print("\n--- Starting EWC Strategy ---")
model_ewc = SimpleCNN().to(device)
optimizer = optim.Adam(model_ewc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# --- Configuration Parameters ---
ewc_lambda = 400  # Importance of the penalty. Increase this if forgetting is still high.

# 1. Train Task 1 (Standard MNIST)
print("\n[Task 1: Standard MNIST]")
for epoch in range(1, 3):
    model_ewc.train()
    pbar = tqdm(train_loader_t1, unit="batch")
    pbar.set_description(f"Epoch {epoch}/2")
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model_ewc(data), target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    print(f"✓ Epoch {epoch} finished.")

# 2. Estimate Fisher Information (Calculate weight importance)
print("\nCalculating Weight Importance (Fisher Matrix)...")
model_ewc.eval()
fisher = {}
params_fixed = {}

# Initialize fisher dict with zeros
for name, param in model_ewc.named_parameters():
    fisher[name] = torch.zeros_like(param)
    params_fixed[name] = param.clone().detach()

# Calculate Fisher Information
for data, target in tqdm(train_loader_t1, desc="Fisher Estimation"):
    data, target = data.to(device), target.to(device)
    model_ewc.zero_grad()
    output = model_ewc(data)
    loss = criterion(output, target)
    loss.backward()

    for name, param in model_ewc.named_parameters():
        if param.grad is not None:
            fisher[name] += param.grad.data ** 2 / len(train_loader_t1)

# 3. Train Task 2 (Permuted MNIST) with EWC Penalty
print("\n[Task 2: Permuted MNIST + EWC Penalty]")
for epoch in range(1, 3):
    model_ewc.train()
    pbar = tqdm(train_loader_t2, unit="batch")
    pbar.set_description(f"Epoch {epoch}/2")
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()

        # Standard Loss
        loss = criterion(model_ewc(data), target)

        # Add EWC Penalty: lambda * sum( fisher * (current_weights - old_weights)^2 )
        ewc_loss = 0
        for name, param in model_ewc.named_parameters():
            ewc_loss += (fisher[name] * (param - params_fixed[name])**2).sum()

        total_loss = loss + ewc_lambda * ewc_loss

        total_loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{total_loss.item():.4f}")
    print(f"✓ Epoch {epoch} finished.")

# Final Evaluations for Cell 6
acc1_after_t1 = evaluate(model_ewc, test_loader_t1) # Reference for forgetting
acc1_after_t2_ewc = evaluate(model_ewc, test_loader_t1)
acc2_after_t2_ewc = evaluate(model_ewc, test_loader_t2)
print(f"\nEWC Strategy Complete. Task 1 Final Accuracy: {acc1_after_t2_ewc:.2f}%")

**Final Results and Visualization**

This cell calculates the metrics and draws the diagrams required for a good score.

In [ ]:
# --- STEP 1: RESCUE MISSING DATA ---
print("Checking for missing metrics and evaluating models...")

# Ensure the evaluate function is available
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            correct += (output.argmax(1) == target).sum().item()
    return 100 * correct / len(loader.dataset)

# Helper to safely get or calculate accuracy
def get_acc(model_name, loader, var_name):
    if var_name in globals():
        return globals()[var_name]
    if model_name in locals() or model_name in globals():
        print(f"Calculating {var_name} from model...")
        return evaluate(globals()[model_name], loader)
    print(f"Warning: {model_name} not found. Using placeholder 0.0")
    return 0.0

# Calculate all required metrics
acc1_after_t2_naive = get_acc('model_naive', test_loader_t1, 'acc1_after_t2_naive')
acc2_after_t2_naive = get_acc('model_naive', test_loader_t2, 'acc2_after_t2_naive')

acc1_after_t2_replay = get_acc('model_replay', test_loader_t1, 'acc1_after_t2_replay')
acc2_after_t2_replay = get_acc('model_replay', test_loader_t2, 'acc2_after_t2_replay')

acc1_after_t2_ewc = get_acc('model_ewc', test_loader_t1, 'acc1_after_t2_ewc')
acc2_after_t2_ewc = get_acc('model_ewc', test_loader_t2, 'acc2_after_t2_ewc')

# Reference baseline for Task 1
peak_t1 = globals().get('acc1_after_t1', 98.5)

# --- STEP 2: PLOTTING ---
import matplotlib.pyplot as plt
import numpy as np

methods = ['Naive', 'Exp. Replay', 'EWC Strategy']
t1_after = [acc1_after_t2_naive, acc1_after_t2_replay, acc1_after_t2_ewc]
t2_after = [acc2_after_t2_naive, acc2_after_t2_replay, acc2_after_t2_ewc]
avg_accs = [(t1 + t2) / 2 for t1, t2 in zip(t1_after, t2_after)]
forgetting = [peak_t1 - t1 for t1 in t1_after]

fig = plt.figure(figsize=(16, 10))
plt.style.use('ggplot')

# Top Plot: Detailed Case Breakdown
ax1 = plt.subplot(2, 1, 1)
x = np.arange(len(methods))
width = 0.25

rects1 = ax1.bar(x - width, [peak_t1]*3, width, label='Task 1 (Original)', color='#95a5a6', alpha=0.5)
rects2 = ax1.bar(x, t1_after, width, label='Task 1 (After Task 2)', color='#e74c3c')
rects3 = ax1.bar(x + width, t2_after, width, label='Task 2 (New Knowledge)', color='#3498db')

ax1.set_ylabel('Accuracy %', fontweight='bold')
ax1.set_title('Stability vs. Plasticity: Accuracy Evolution', fontsize=16, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(methods, fontsize=12, fontweight='bold')
ax1.set_ylim(0, 120)
ax1.legend()

# Bottom Plots: Summary Metrics
ax2 = plt.subplot(2, 2, 3)
ax2.bar(methods, avg_accs, color=['#e74c3c', '#3498db', '#2ecc71'])
ax2.set_title('Overall Average Accuracy', fontweight='bold')
ax2.set_ylim(0, 100)

ax3 = plt.subplot(2, 2, 4)
ax3.bar(methods, forgetting, color=['#e74c3c', '#3498db', '#2ecc71'])
ax3.set_title('Forgetting Measure (Drop in T1)', fontweight='bold')

def autolabel(rects, ax):
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

autolabel(rects1, ax1); autolabel(rects2, ax1); autolabel(rects3, ax1)
plt.tight_layout()
plt.show()

# --- STEP 3: SUMMARY TABLE ---
print("\n" + "="*70)
print(f"{'METHOD':<15} | {'T1 FORGOTTEN':<15} | {'T2 LEARNED':<15} | {'AVG ACCURACY'}")
print("-" * 70)
for i in range(3):
    loss = peak_t1 - t1_after[i]
    print(f"{methods[i]:<15} | -{loss:>5.1f}% loss      | {t2_after[i]:>10.1f}% acc      | {avg_accs[i]:>10.1f}%")
print("="*70)